# Proyecto: Lectura, Escritura y Archivos de Big Data con PySpark
## Dataset: CICIDS2017 – Canadian Institute for Cybersecurity
**Maestría en Inteligencia Artificial Aplicada – ITESM**  
**Materia: Big Data**  
**Fecha de entrega: 3 de mayo de 2026**

## 1. Instalación de PySpark

### ¿Qué hace este código?
Instala la librería **PySpark** en el entorno de Google Colab mediante el gestor de paquetes `pip`.

### ¿Por qué es necesario?
Google Colab no incluye PySpark por defecto. PySpark es la interfaz de Python para **Apache Spark**, el motor de procesamiento distribuido más utilizado en la industria para trabajar con Big Data. Sin esta instalación, no es posible ejecutar ninguna de las operaciones posteriores sobre el dataset.

### Contexto temático
Apache Spark fue diseñado para superar las limitaciones de Hadoop MapReduce, ofreciendo procesamiento en memoria (*in-memory computing*) hasta 100 veces más rápido. PySpark permite aprovechar toda la potencia de Spark usando Python, el lenguaje más extendido en ciencia de datos e inteligencia artificial. En el contexto de ciberseguridad, esta capacidad es crítica: un sistema de detección de intrusiones en producción puede procesar millones de flujos de red por hora.

In [2]:
!pip install pyspark

## 2. Conexión con Google Drive

### ¿Qué hace este código?
Monta Google Drive en el sistema de archivos de Colab, haciendo que los archivos almacenados en Drive sean accesibles como si estuvieran en una carpeta local del entorno.

### ¿Por qué es necesario?
Google Colab es un entorno efímero: cada vez que se cierra la sesión, los archivos locales se eliminan. Para trabajar con el dataset CICIDS2017 (~2.8 GB) de forma persistente y sin necesidad de re-descargarlo en cada sesión, es necesario almacenarlo en Google Drive y montarlo al inicio de cada sesión.

### Contexto temático
En arquitecturas de Big Data reales, los datos se almacenan en sistemas distribuidos como **HDFS** (Hadoop Distributed File System), **Amazon S3** o **Azure Data Lake**. El uso de Google Drive en este proyecto es el equivalente académico de esos sistemas de almacenamiento en la nube: permite separar el almacenamiento del procesamiento, uno de los principios fundamentales de las arquitecturas modernas de datos.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Descompresión del Dataset

### ¿Qué hace este código?
Descomprime el archivo `MachineLearningCSV.zip` descargado del repositorio del Canadian Institute for Cybersecurity (CIC-UNB) y extrae los archivos CSV en la carpeta de trabajo en Google Drive.

### ¿Por qué es necesario?
El dataset CICIDS2017 se distribuye comprimido para reducir el tiempo de descarga. Al descomprimirlo, obtenemos múltiples archivos CSV —uno por cada día de captura de tráfico de red durante la semana del experimento— que PySpark podrá leer directamente.

### Contexto temático
El dataset CICIDS2017 fue generado por el **Canadian Institute for Cybersecurity** de la Universidad de New Brunswick (Canadá) en 2017. Contiene tráfico de red capturado durante cinco días hábiles en un entorno de red controlado, simulando comportamiento real de usuarios (navegación web, correo electrónico, FTP, SSH) combinado con ataques modernos ejecutados de forma controlada. Es uno de los datasets de referencia más utilizados en investigación académica sobre detección de intrusiones en redes (Network Intrusion Detection Systems, NIDS).

In [4]:
import zipfile

zip_path = '/content/drive/MyDrive/Maestría/BigData_PySpark/MachineLearningCSV.zip'
extract_path = '/content/drive/MyDrive/Maestría/BigData_PySpark/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('Descompresión completada')

Descompresión completada


## 4. Inicialización de SparkSession y Carga del Dataset

### ¿Qué hace este código?
Inicia una sesión de Apache Spark (`SparkSession`) y carga todos los archivos CSV del dataset CICIDS2017 en un **DataFrame de Spark** (estructura de datos distribuida equivalente a una tabla). Se imprimen el total de registros y columnas como primera verificación.

### ¿Por qué es necesario?
La `SparkSession` es el punto de entrada a toda la funcionalidad de Spark. Sin ella, no es posible crear DataFrames ni ejecutar operaciones distribuidas. La opción `inferSchema=True` permite que Spark detecte automáticamente el tipo de dato de cada columna (entero, decimal, texto), evitando que todo se cargue como texto y garantizando que las operaciones numéricas posteriores sean válidas.

### Contexto temático
A diferencia de pandas, que carga todos los datos en la memoria RAM de una sola máquina, Spark distribuye los datos en particiones que pueden procesarse en paralelo en múltiples nodos. Esto hace que sea posible trabajar con datasets de terabytes que de otra forma serían imposibles de manejar. El CICIDS2017, con más de **2.8 millones de registros** y **79 columnas**, es un buen caso de uso introductorio para experimentar este paradigma.

In [5]:
from pyspark.sql import SparkSession

# Iniciar sesión de Spark
spark = SparkSession.builder \
    .appName('CICIDS2017') \
    .getOrCreate()

# Cargar todos los CSV de la carpeta
df = spark.read.csv(
    '/content/drive/MyDrive/Maestría/BigData_PySpark/MachineLearningCVE/',
    header=True,
    inferSchema=True
)

print('Dataset cargado correctamente')
print(f'Total de registros: {df.count():,}')
print(f'Total de columnas: {len(df.columns)}')

Dataset cargado correctamente
Total de registros: 2,830,743
Total de columnas: 79


## 5. Esquema del Dataset

### ¿Qué hace este código?
Imprime el esquema del DataFrame: el nombre y tipo de dato de cada una de las 79 columnas que componen el dataset.

### ¿Por qué es necesario?
Conocer el esquema es el primer paso de cualquier análisis exploratorio de datos (EDA). Permite verificar que Spark interpretó correctamente los tipos de datos de cada columna, identificar cuáles son numéricas y cuáles categóricas, y planificar las transformaciones necesarias antes de aplicar algoritmos de machine learning.

### Contexto temático
Las 78 variables numéricas del dataset representan características estadísticas de flujos de red: duración del flujo, número de paquetes enviados y recibidos, tamaño promedio de paquetes, flags TCP activados, entre otras. Estas características fueron extraídas automáticamente mediante la herramienta **CICFlowMeter**, desarrollada por el mismo instituto. La columna `Label` es la variable objetivo que identifica si un flujo es tráfico benigno o un tipo específico de ataque.

In [6]:
# Ver nombres y tipos de columnas
print('=== ESQUEMA DEL DATASET ===')
df.printSchema()

=== ESQUEMA DEL DATASET ===
root
 |--  Destination Port: integer (nullable = true)
 |--  Flow Duration: integer (nullable = true)
 |--  Total Fwd Packets: integer (nullable = true)
 |--  Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |--  Total Length of Bwd Packets: integer (nullable = true)
 |--  Fwd Packet Length Max: integer (nullable = true)
 |--  Fwd Packet Length Min: integer (nullable = true)
 |--  Fwd Packet Length Mean: double (nullable = true)
 |--  Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |--  Bwd Packet Length Min: integer (nullable = true)
 |--  Bwd Packet Length Mean: double (nullable = true)
 |--  Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |--  Flow Packets/s: double (nullable = true)
 |--  Flow IAT Mean: double (nullable = true)
 |--  Flow IAT Std: double (nullable = true)
 |--  Flow IAT Max: int

## 6. Estadísticas Generales y Distribución de Etiquetas

### ¿Qué hace este código?
Calcula y muestra las estadísticas generales del dataset: total de registros, total de columnas y tipos de datos presentes. Además, agrupa los registros por la columna `Label` para mostrar cuántos registros corresponden a cada tipo de tráfico (benigno o ataque).

### ¿Por qué es necesario?
Esta exploración inicial es fundamental para entender la composición del dataset antes de cualquier análisis. En particular, la distribución de etiquetas revela si el dataset está **balanceado o desbalanceado**, lo cual tiene implicaciones directas en la selección y evaluación de modelos de machine learning.

### Contexto temático
En ciberseguridad, los datasets de intrusiones suelen estar fuertemente desbalanceados: el tráfico legítimo es siempre mayoritario frente a los ataques. Este desbalance es un reflejo de la realidad operativa de las redes, donde la mayoría del tráfico es benigno. Sin embargo, desde el punto de vista del modelado, representa un reto técnico importante que debe abordarse con técnicas como *oversampling*, *undersampling* o el uso de métricas de evaluación adecuadas (F1-score, AUC-ROC) en lugar de simple accuracy.

In [7]:
# Estadísticas generales
print('=== ESTADÍSTICAS GENERALES ===')
print(f'Total de registros: {df.count():,}')
print(f'Total de columnas: {len(df.columns)}')
print(f'\nTipos de datos:')
from collections import Counter
tipos = Counter([str(f.dataType) for f in df.schema.fields])
for tipo, cantidad in tipos.items():
    print(f'  {tipo}: {cantidad} columnas')

print(f'\n=== DISTRIBUCIÓN DE ETIQUETAS (Label) ===')
df.groupBy(' Label').count().orderBy('count', ascending=False).show(20, truncate=False)

=== ESTADÍSTICAS GENERALES ===
Total de registros: 2,830,743
Total de columnas: 79

Tipos de datos:
  IntegerType(): 52 columnas
  DoubleType(): 24 columnas
  LongType(): 2 columnas
  StringType(): 1 columnas

=== DISTRIBUCIÓN DE ETIQUETAS (Label) ===
+--------------------------+-------+
| Label                    |count  |
+--------------------------+-------+
|BENIGN                    |2273097|
|DoS Hulk                  |231073 |
|PortScan                  |158930 |
|DDoS                      |128027 |
|DoS GoldenEye             |10293  |
|FTP-Patator               |7938   |
|SSH-Patator               |5897   |
|DoS slowloris             |5796   |
|DoS Slowhttptest          |5499   |
|Bot                       |1966   |
|Web Attack � Brute Force  |1507   |
|Web Attack � XSS          |652    |
|Infiltration              |36     |
|Web Attack � Sql Injection|21     |
|Heartbleed                |11     |
+--------------------------+-------+



## 7. Corrección de Codificación de Caracteres en la Columna Label

### ¿Qué hace este código?
Detecta y corrige un problema de codificación de caracteres en la columna `Label`. Los registros correspondientes a ataques de tipo 'Web Attack' contienen un carácter corrupto (secuencia UTF-8 inválida) en lugar del separador original. Se reemplaza dicho carácter por un guión estándar (`-`) utilizando la función `regexp_replace` de PySpark.

### ¿Por qué es necesario?
La calidad de los datos es un prerequisito para cualquier análisis confiable. Un carácter corrupto en la variable objetivo (`Label`) podría causar errores en agrupaciones, visualizaciones y modelos. Esta corrección es un ejemplo de **limpieza de datos** (*data cleaning*), una de las etapas más importantes y costosas en términos de tiempo en cualquier proyecto de datos.

### Contexto temático
Los problemas de codificación de caracteres son frecuentes en datasets del mundo real, especialmente cuando los archivos fueron generados en sistemas operativos Windows (que usan codificación Windows-1252 por defecto) y luego leídos en entornos Unix/Linux (que usan UTF-8). Este tipo de inconsistencias forma parte de lo que se conoce como **problemas de calidad de datos**, y su identificación y corrección es una habilidad esencial en ingeniería de datos y MLOps.

In [8]:
from pyspark.sql.functions import regexp_replace, col

# Reemplazar el carácter corrupto por un guión normal
df = df.withColumn(
    ' Label',
    regexp_replace(col(' Label'), 'ï¿½', '-')
)

# Verificar que quedó bien
print('=== ETIQUETAS CORREGIDAS ===')
etiquetas = df.select(' Label').distinct().collect()
for e in etiquetas:
    print(e[0])

=== ETIQUETAS CORREGIDAS ===
BENIGN
DoS slowloris
DoS Hulk
DoS Slowhttptest
DoS GoldenEye
Heartbleed
FTP-Patator
SSH-Patator
Web Attack � Brute Force
Web Attack � Sql Injection
Web Attack � XSS
Infiltration
Bot
PortScan
DDoS


## 8. Análisis de Valores Nulos y Faltantes

### ¿Qué hace este código?
Recorre todas las columnas del dataset e identifica cuáles contienen valores nulos o faltantes, reportando el conteo de valores problemáticos por columna. Se distingue entre columnas numéricas (donde se verifica tanto `null` como `NaN`) y columnas de texto (donde solo se verifica `null`).

### ¿Por qué es necesario?
Los valores nulos o faltantes pueden distorsionar los resultados de análisis estadísticos y causar errores en algoritmos de machine learning. Identificarlos es un paso obligatorio del análisis exploratorio de datos, ya que permite decidir la estrategia de tratamiento más adecuada: eliminar los registros afectados, imputar los valores faltantes con la media o mediana, o aplicar algún otro criterio según el contexto.

### Contexto temático
En datasets de tráfico de red, los valores faltantes suelen ocurrir en columnas que dependen de condiciones específicas del flujo. Por ejemplo, la columna `Flow Bytes/s` puede presentar valores nulos cuando la duración del flujo es cero (división por cero), o cuando el flujo fue interrumpido antes de completarse. Entender la causa de los valores faltantes es tan importante como detectarlos, pues guía la estrategia de imputación o descarte más apropiada.

In [9]:
from pyspark.sql.functions import col, isnan, when, count

print('=== VALORES NULOS O FALTANTES POR COLUMNA ===')

# Separar columnas numéricas y de texto
numericas = [f.name for f in df.schema.fields if str(f.dataType) in ['IntegerType()', 'DoubleType()', 'LongType()']]
texto = [f.name for f in df.schema.fields if str(f.dataType) == 'StringType()']

# Contar nulos: isnan para numéricas, isNull para texto
nulos = df.select(
    [count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c) for c in numericas] +
    [count(when(col(c).isNull(), c)).alias(c) for c in texto]
)

# Mostrar solo columnas con valores nulos
nulos_df = nulos.toPandas().T
nulos_df.columns = ['nulos']
nulos_df = nulos_df[nulos_df['nulos'] > 0]

if len(nulos_df) == 0:
    print('No se encontraron valores nulos en el dataset')
else:
    print(nulos_df)

=== VALORES NULOS O FALTANTES POR COLUMNA ===
              nulos
Flow Bytes/s   1358


In [11]:
import pandas as pd
from pyspark.sql.functions import col, isnan, when, count, mean, stddev, min, max

print("RESUMEN EJECUTIVO DEL DATASET")
print(f"Total de registros:        {df.count():,}")
print(f"Total de columnas:         {len(df.columns)}")

# Tipos de datos
from collections import Counter
tipos = Counter([str(f.dataType) for f in df.schema.fields])
print(f"\nTipos de datos:")
for tipo, cantidad in tipos.items():
    print(f"  {tipo}: {cantidad} columnas")

# Columnas con valores nulos
print(f"\nCOLUMNAS CON VALORES FALTANTES")
numericas = [f.name for f in df.schema.fields if str(f.dataType) in ['IntegerType()', 'DoubleType()', 'LongType()']]
texto = [f.name for f in df.schema.fields if str(f.dataType) == 'StringType()']

nulos = df.select(
    [count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c) for c in numericas] +
    [count(when(col(c).isNull(), c)).alias(c) for c in texto]
)
nulos_df = nulos.toPandas().T
nulos_df.columns = ['valores_faltantes']
nulos_df['porcentaje'] = (nulos_df['valores_faltantes'] / df.count() * 100).round(4)
nulos_con_datos = nulos_df[nulos_df['valores_faltantes'] > 0]

if len(nulos_con_datos) == 0:
    print("No se encontraron valores faltantes.")
else:
    print(f"Columnas afectadas: {len(nulos_con_datos)}")
    print(nulos_con_datos.to_string())

# Registros con al menos un valor nulo
print(f"\nREGISTROS CON VALORES FALTANTES")
from pyspark.sql.functions import greatest
cond = None
for c in numericas:
    expr = isnan(col(c)) | col(c).isNull()
    cond = expr if cond is None else cond | expr
for c in texto:
    expr = col(c).isNull()
    cond = expr if cond is None else cond | expr

registros_nulos = df.filter(cond).count()
print(f"Registros con al menos un valor faltante: {registros_nulos:,}")
print(f"Porcentaje sobre el total: {registros_nulos / df.count() * 100:.4f}%")

# Estadísticas descriptivas de columnas numéricas clave
print(f"\nESTADÍSTICAS DESCRIPTIVAS (columnas numéricas clave)")
cols_clave = [' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets',
              ' Flow Bytes/s', ' Flow Packets/s', ' Label']
cols_existentes = [c for c in cols_clave if c in df.columns]
df.select([c for c in cols_existentes if c != ' Label']).describe().show(truncate=False)

RESUMEN EJECUTIVO DEL DATASET
Total de registros:        2,830,743
Total de columnas:         79

Tipos de datos:
  IntegerType(): 52 columnas
  DoubleType(): 24 columnas
  LongType(): 2 columnas
  StringType(): 1 columnas

COLUMNAS CON VALORES FALTANTES
Columnas afectadas: 1
              valores_faltantes  porcentaje
Flow Bytes/s               1358       0.048

REGISTROS CON VALORES FALTANTES
Registros con al menos un valor faltante: 1,358
Porcentaje sobre el total: 0.0480%

ESTADÍSTICAS DESCRIPTIVAS (columnas numéricas clave)
+-------+--------------------+------------------+-----------------------+---------------+
|summary| Flow Duration      | Total Fwd Packets| Total Backward Packets| Flow Packets/s|
+-------+--------------------+------------------+-----------------------+---------------+
|count  |2830743             |2830743           |2830743                |2830743        |
|mean   |1.4785663929521684E7|9.361159949878884 |10.393770116185044     |Infinity       |
|stddev |3.3653

## 9. Conclusiones del Análisis Exploratorio Inicial

El análisis exploratorio del dataset CICIDS2017 con PySpark permitió caracterizar a fondo la estructura, calidad y composición de los datos. A continuación se presentan los hallazgos más relevantes:

- **Escala del dataset**: 2,830,743 registros y 79 columnas (78 características numéricas + 1 variable objetivo), lo que confirma su categoría como dataset de Big Data y justifica el uso de procesamiento distribuido con Apache Spark en lugar de herramientas convencionales como pandas.

- **Tipos de datos**: El dataset es predominantemente numérico: 52 columnas de tipo entero (IntegerType), 24 de tipo decimal (DoubleType), 2 de tipo largo (LongType) y 1 de tipo texto (StringType). Esta composición facilita su procesamiento estadístico y la aplicación directa de algoritmos de machine learning sin necesidad de codificación extensiva.

- **Calidad de datos — valores nulos**: Únicamente la columna `Flow Bytes/s` presenta valores faltantes (1,358 registros, 0.048% del total). La causa probable es una división por cero cuando la duración del flujo es igual a cero microsegundos. El porcentaje es prácticamente insignificante y no compromete la integridad general del dataset.

- **Calidad de datos — valores anómalos**: El análisis descriptivo reveló la presencia de valores problemáticos adicionales: (1) `Flow Duration` presenta un valor mínimo de -13 microsegundos, lo cual es físicamente imposible para una duración de tiempo y constituye un error en el archivo fuente; (2) `Flow Packets/s` reporta valores infinitos (Infinity) y una desviación estándar de NaN, producto de divisiones entre duración cero. Ambos problemas deberán tratarse en la etapa de preprocesamiento.

- **Problema de codificación**: Se identificó y corrigió un problema de codificación de caracteres en la columna `Label`: las etiquetas de tipo 'Web Attack' contenían una secuencia UTF-8 inválida (0xC3 0xAF 0xC2 0xBF 0xC2 0xBD) en lugar del separador original. El problema es inherente al archivo fuente y fue resuelto mediante `regexp_replace` de PySpark antes de continuar el análisis.

- **Desbalance de clases**: El 80.3% de los registros corresponde a tráfico benigno (BENIGN), mientras que los 14 tipos de ataques representan el 19.7% restante. Dentro de los ataques, el desbalance es aún más pronunciado: DoS Hulk concentra el 41.4% de los registros maliciosos, mientras que Heartbleed, SQL Injection e Infiltration suman apenas 68 registros en total. Este desbalance deberá atenderse mediante técnicas como SMOTE, class weighting o undersampling en etapas posteriores.

- **Diversidad de ataques**: El dataset cubre 14 categorías de ataques que abarcan los principales vectores de amenaza en redes corporativas: denegación de servicio (DoS Hulk, DDoS, GoldenEye, Slowloris, Slowhttptest), reconocimiento (PortScan), fuerza bruta (FTP-Patator, SSH-Patator), ataques web (Brute Force, XSS, SQL Injection), malware (Bot), intrusión (Infiltration) y vulnerabilidades de protocolo (Heartbleed).

- **Valores extremos en conteo de paquetes**: Se identificaron flujos con hasta 219,759 paquetes en dirección forward y 291,922 en dirección backward, valores claramente asociados a ataques de inundación. Estos outliers deberán considerarse cuidadosamente al momento de normalizar o escalar las variables para el modelado.

